# Laporan Praktikum 1 dan 2
Praktikum ini berisi 5 proses pemodelan machine learning yaitu EDA, Scrapping, Regresi Linear, Decision Tree dan Naive Bayes

## 1.1 Exploratory Data Analysis

### Import Library

In [ ]:
# Maftuh Faisal_202431163
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re

In [ ]:
# Maftuh Faisal_202431163
df = pd.read_csv('manga_sales.csv')
df.info() 
df.head(10)

In [ ]:
# Maftuh Faisal_202431163
def extract_number(x):
    if pd.isna(x): return np.nan
    match = re.search(r'[-+]?\d*\.?\d+', str(x))
    return float(match.group()) if match else np.nan

num_cols = ['age', 'unit_price', 'quantity', 'total_amount', 'customer_rating', 'delivery_days']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].apply(extract_number)

cat_col = ['gender', 'city', 'genre', 'format', 'platform', 'payment_method', 'membership_tier']
for col in cat_col:
    if col in df.columns:
        df[col] = df[col].astype('category')

df.dtypes

In [ ]:
# Maftuh Faisal_202431163
df.describe(include="all")

In [ ]:
# Maftuh Faisal_202431163
df.isna().sum() 

In [ ]:
# Maftuh Faisal_202431163
for col in cat_col:
    if col in df.columns:
        print(f"=== {col} ===")
        print(df[col].unique())
        print("\n")

In [ ]:
# Maftuh Faisal_202431163
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

df.isna().sum()

In [ ]:
# Maftuh Faisal_202431163
print("Jumlah baris duplikat sebelum dihapus:", df.duplicated().sum())

df = df.drop_duplicates()
df.info()

In [ ]:
# Maftuh Faisal_202431163
plt.figure(figsize=(8, 5))
sns.countplot(x="is_returned", data=df, color='teal')
plt.title("Distribusi Barang yang Direturn (is_returned)")
plt.show()

In [ ]:
# Maftuh Faisal_202431163
plt.figure(figsize=(10, 5))
df['genre'].value_counts().plot(kind='bar', color='teal', edgecolor='none')
plt.title("Jumlah Transaksi per Genre Manga")
plt.xlabel("Genre")
plt.ylabel("Jumlah Transaksi")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Maftuh Faisal_202431163
num_df = df.select_dtypes(include=['float64', 'int64'])
plt.figure(figsize=(10, 6))
sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm")
plt.title("Korelasi Antar Variabel Numerik")
plt.show()

## 1.2 Scrapping

### Import Library

In [ ]:
# Maftuh Faisal_202431163
import requests
import csv
from bs4 import BeautifulSoup

In [ ]:
# Maftuh Faisal_202431163
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

base_url = "https://myanimelist.net/topanime.php?limit={}"
pages_html = []

for limit in [0, 50, 100, 150]:
    url = base_url.format(limit)
    response = requests.get(url, headers=headers)

    print(f"URL: {url} -> status: {response.status_code}")

    if response.status_code != 200:
        raise Exception(f"Gagal mengambil {url}. Status: {response.status_code}")

    pages_html.append(response.text)

print(f"\nTotal halaman berhasil diambil: {len(pages_html)}")

In [ ]:
# Maftuh Faisal_202431163
soups = [BeautifulSoup(html, "html.parser") for html in pages_html]

anime_blocks = []
for soup in soups:
    rows = soup.find_all("tr", class_="ranking-list")
    anime_blocks.extend(rows)

print(f"Jumlah baris anime yang ditemukan: {len(anime_blocks)}")

if len(anime_blocks) < 200:
    print("Data kurang dari 200. Tambahkan limit halaman lain.")
else:
    print("Jumlah data memenuhi syarat minimal 200.")

In [ ]:
# Maftuh Faisal_202431163
import re

result = []

for block in anime_blocks:
    
    title_element = block.find("h3", class_="anime_ranking_h3")
    title_name = title_element.get_text(strip=True) if title_element else None

    info_element = block.find("div", class_="information")
    info_text = info_element.get_text(" ", strip=True) if info_element else ""

    anime_type = None
    episodes = None
    members_value = None

    if info_text:
    
        if "(" in info_text and ")" in info_text:
            anime_type = info_text.split("(")[0].strip()
            inside = info_text.split("(")[1].split(")")[0]
            if "eps" in inside.lower():
                episodes = inside.lower().replace("eps", "").strip()

        match = re.search(r"([\d,]+)\s*members", info_text)
        if match:
            members_value = match.group(1).replace(",", "")

    score_element = block.find("span", class_="score-label")
    score_value = score_element.get_text(strip=True) if score_element else None

    result.append({
        "title": title_name,
        "type": anime_type,
        "episodes": episodes,
        "score": score_value,
        "members": members_value,
    })

for r in result[:5]:
    print(r)

In [ ]:
# Maftuh Faisal_202431163
for item in result:
    print(
        f"Title: {item['title']} - "
        f"Type: {item['type']} - "
        f"Episodes: {item['episodes']} - "
        f"Score: {item['score']} - "
        f"Members: {item['members']}"
    )

In [ ]:
# Maftuh Faisal_202431163
with open("anime_top.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["title", "type", "episodes", "score", "members"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for item in result:
        writer.writerow(item)

print("Data disimpan ke anime_top.csv")

In [ ]:
# Maftuh Faisal_202431163
import pandas as pd

db = pd.read_csv("anime_top.csv")

db["episodes"] = pd.to_numeric(db["episodes"], errors="coerce")
db["score"] = pd.to_numeric(db["score"], errors="coerce")
db["members"] = pd.to_numeric(db["members"], errors="coerce")

print(db.dtypes)

db.to_csv("anime_top_clean.csv", index=False)
print("Data bersih disimpan ke anime_top_clean.csv")

## 1.3 Regresi Linear

### Import Library

In [ ]:
# Maftuh Faisal_202431163
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split as tts
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Maftuh Faisal_202431163
df = pd.read_csv('anime_top_clean.csv')
df

In [ ]:
# Maftuh Faisal_202431163
df.info()

In [ ]:
# Maftuh Faisal_202431163
import seaborn as sns

num_df = df.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(10, 6))
sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm")
plt.title("Korelasi antar atribut numerik")
plt.show()

In [ ]:
# Maftuh Faisal_202431163
db_clean = df.dropna(subset=["episodes", "members", "score"]).copy()

db_clean["episodes"] = pd.to_numeric(db_clean["episodes"], errors="coerce")
db_clean["members"]  = pd.to_numeric(db_clean["members"],  errors="coerce")
db_clean["score"]    = pd.to_numeric(db_clean["score"],    errors="coerce")

db_clean = db_clean.dropna(subset=["episodes", "members", "score"])

x = db_clean[["episodes", "members"]]  
y = db_clean["score"]                    

print("Jumlah data setelah dropna:", len(db_clean))
print("Contoh x:")
print(x.head())
print("\nContoh y:")
print(y.head())

In [ ]:
# Maftuh Faisal_202431163
x_train, x_test, y_train, y_test = tts(x, y, test_size=0.2, random_state=0)

print("Jumlah data latih:", len(x_train))
print("Jumlah data uji  :", len(x_test))

In [ ]:
# Maftuh Faisal_202431163
model = LinearRegression()
model.fit(x_train, y_train)

In [ ]:
# Maftuh Faisal_202431163
y_pred = model.predict(x_test)
y_pred

In [ ]:
# Maftuh Faisal_202431163
print("coefficient :", model.coef_)
print("intercept   :", model.intercept_)
print("MSE         :", mean_squared_error(y_test, y_pred))
print("R2 Score    :", r2_score(y_test, y_pred))

In [ ]:
# Maftuh faisal_202431163
a = model.intercept_
b1, b2 = model.coef_

print("Persamaan regresi:")
print(f"Y = {a:.4f} + {b1:.6f}·x1 + {b2:.6f}·x2")
print()
print("Dengan:")
print("  Y  = score (rating anime)")
print("  x1 = episodes (jumlah episode)")
print("  x2 = members (jumlah member/popularitas)")
print()
print(f"Interpretasi:")
print(f"  - Setiap tambahan 1 episode  → score berubah {b1:+.6f}")
print(f"  - Setiap tambahan 1 member   → score berubah {b2:+.8f}")

In [ ]:
# Maftuh Faisal_202431163
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Nilai asli y_test (score)")
plt.ylabel("Nilai prediksi y_test (score)")
plt.title("Nilai asli vs Nilai prediksi")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--')
plt.show()

## 2.1 Decision Tree

### Import Library

In [ ]:
# Maftuh Faisal_202431163
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Maftuh Faisal_202431163
df = pd.read_csv('anime_top_clean.csv', sep=",")
df

In [ ]:
# Maftuh Faisal_202431163
df.info()

In [ ]:
# Maftuh Faisal_202431163
df.describe()

In [ ]:
# Maftuh Faisal_202431163
df.isna().sum()

In [ ]:
# Maftuh Faisal_202431163
df['type'].unique()

In [ ]:
# Maftuh Faisal_202431163
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])
print(df['type'].unique())

for i, label in enumerate(le.classes_):
    print(f"{i} = {label}")

In [ ]:
# Maftuh Faisal_202431163
df

In [ ]:
# Maftuh Faisal_202431163
x = df.drop(['title', 'type'], axis=1)
x

In [ ]:
# Maftuh Faisal_202431163
y = df['type']
y

In [ ]:
# Maftuh Faisal_202431163
x = np.array(x, dtype=np.float64)
y = np.array(y, dtype=np.int64)

print("Shape x:", x.shape, "| dtype:", x.dtype)
print("Shape y:", y.shape, "| dtype:", y.dtype)

In [ ]:
# Maftuh Faisal_202431163
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)

In [ ]:
# Maftuh Faisal_202431163
from sklearn.tree import DecisionTreeClassifier

classifier = DecisionTreeClassifier()
classifier.fit(x_train, y_train)

In [ ]:
# Maftuh Faisal_202431163
from sklearn import tree

plt.subplots(figsize=(15, 10))
tree.plot_tree(
    classifier,
    feature_names=['episodes', 'members', 'score'],
    class_names=le.classes_.tolist(),
    filled=True,
    fontsize=10
)
plt.show()

In [ ]:
# Maftuh Faisal_202431163
depth = classifier.get_depth()
depth

In [ ]:
# Maftuh Faisal_202431163
from sklearn.metrics import accuracy_score

y_pred = classifier.predict(x_test)
akurasi = accuracy_score(y_test, y_pred)
print(akurasi)

In [ ]:
# Maftuh Faisal_202431163
y_pred_new = classifier.predict([[1, 500000, 8.5]])
print(y_pred_new)
print("Prediksi kelas:", le.inverse_transform(y_pred_new))

In [ ]:
# Maftuh Faisal_202431163
from sklearn.metrics import classification_report
semua_label = range(len(le.classes_))
print(classification_report(
    y_test, 
    y_pred, 
    labels=semua_label,          
    target_names=le.classes_, 
    zero_division=0
))

## 2.2 Naive Bayes

### Import Library

In [ ]:
# Maftuh Faisal_202431163
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Maftuh Faisal_202431163
data = pd.read_csv("manga_dataset.csv")

In [ ]:
# Maftuh Faisal_202431163
data.head()

In [ ]:
# Maftuh Faisal_202431163
data.info()

In [ ]:
# Maftuh Faisal_202431163
data.describe()

In [ ]:
# Maftuh Faisal_202431163
drop_cols = ['mal_id', 'title', 'title_english', 'title_japanese', 'synopsis', 'image_url', 'published_from', 'published_to']
data = data.drop(columns=drop_cols)

for col in data.columns:
    if pd.api.types.is_numeric_dtype(data[col]):
        data[col] = data[col].fillna(data[col].median())
    else:
        data[col] = data[col].astype(str).fillna('Unknown')

In [ ]:
# Maftuh Faisal_202431163
numerical_features = [feature for feature in data.columns if data[feature].dtypes != 'str']
print("Jumlah fitur numerik = ", len(numerical_features))
print("Daftar fitur numerik = ", numerical_features)
display(data[numerical_features].head())

In [ ]:
# Maftuh Faisal_202431163
categorical_features = [feature for feature in data.columns if data[feature].dtypes == 'str']
print("Jumlah fitur kategorikal = ", len(categorical_features))
print("Daftar fitur kategorikal = ", categorical_features)
display(data[categorical_features].head())

In [ ]:
# Maftuh Faisal_202431163
from sklearn.preprocessing import LabelEncoder

le_status = LabelEncoder()

le_status.fit(data['status'].astype(str)) 

for index, label in enumerate(le_status.classes_):
    print(f"Angka {index} = {label}")

In [ ]:
# Maftuh Faisal_202431163
from sklearn.preprocessing import LabelEncoder

for feature in categorical_features:
    le = LabelEncoder()
    
    data[feature] = le.fit_transform(data[feature].astype(str))

In [ ]:
# Maftuh Faisal_202431163
data.head()

In [ ]:
# Maftuh Faisal_202431163
x = data.drop(['status'], axis=1)
y = data['status']

print(x.head())
print(y.head())

In [ ]:
# Maftuh Faisal_202431163
from sklearn.model_selection import train_test_split as tts

x_train, x_test, y_train, y_test = tts(x, y, test_size = 0.25, random_state = 0)

In [ ]:
# Maftuh Faisal_202431163
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(x_train, y_train)

In [ ]:
# Maftuh Faisal_202431163
y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)

In [ ]:
# Maftuh Faisal_202431163
from sklearn.metrics import confusion_matrix, classification_report

print("Evaluasi Training")
cm_train = confusion_matrix(y_train, y_pred_train)
print(cm_train)
print(classification_report(y_train, y_pred_train))

plt.figure(figsize = (6, 4))
sns.heatmap(cm_train, annot = True, fmt='d', cmap = 'Blues', xticklabels = np.unique(y), yticklabels = np.unique(y))
plt.title("Confusion Matrix - Training")
plt.xlabel("Label Terprediksi")
plt.ylabel("Label Asli")
plt.show()

In [ ]:
# Maftuh Faisal_202431163
from sklearn.metrics import confusion_matrix, classification_report

print("Evaluasi Testing")
cm_test = confusion_matrix(y_test, y_pred_test)
print(cm_test)
print(classification_report(y_test, y_pred_test))

plt.figure(figsize = (6, 4))
sns.heatmap(cm_test, annot = True, fmt='d', cmap = 'Greens', xticklabels = np.unique(y), yticklabels = np.unique(y))
plt.title("Confusion Matrix - Testing")
plt.xlabel("Label Terprediksi")
plt.ylabel("Label Asli")
plt.show()

In [ ]:
# Maftuh Faisal_202431163
new_data = {}
for col in x.columns:
    if col in numerical_features:
        new_data[col] = [data[col].median()]
    else:
        new_data[col] = [data[col].mode()[0]]

new_df = pd.DataFrame(new_data)
prediction = model.predict(new_df)
print("Hasil Prediksi Data Baru:", prediction[0])